# NB04 Hash functions — Chapter 4

A chapter notebook for *Blockchain for Lawyers: A Reading Through Technology and Economics*. The code is the same as in the text.

- All code here is a **toy implementation** meant to make a principle visible. Do not use it with real assets, real private keys or real transactions.
- No external API is called, and everything runs without a network. Cells that need the bundled snapshot data carry that data inside the cell.
- Standard library only (run and checked on Python 3.13). There is no install cell.
- The outputs are the editor's own run, stored as produced. Running the cells again should give the same values (random seeds are fixed).

In [ ]:
# NB04-1: taking SHA-256 fingerprints — a one-character difference
import hashlib

def sha256_hex(s: str) -> str:
    """Encode a string as UTF-8 bytes and return its SHA-256 digest as 64 hexadecimal characters."""
    return hashlib.sha256(s.encode("utf-8")).hexdigest()

for s in ["hello", "hellp", "Hello", "hello "]:
    print(f"{s!r:10} -> {sha256_hex(s)}")

def differing_bits(h1: str, h2: str) -> int:
    """Number of bits that differ between two hexadecimal digests (Hamming distance)."""
    return bin(int(h1, 16) ^ int(h2, 16)).count("1")

print()
print("hello vs hellp : differing bits", differing_bits(sha256_hex("hello"), sha256_hex("hellp")), "/ 256")
print("hello vs Hello : differing bits", differing_bits(sha256_hex("hello"), sha256_hex("Hello")), "/ 256")
print("hello vs 'hello ': differing bits", differing_bits(sha256_hex("hello"), sha256_hex("hello ")), "/ 256")

'hello'    -> 2cf24dba5fb0a30e26e83b2ac5b9e29e1b161e5c1fa7425e73043362938b9824
'hellp'    -> fdd7585e08c4e2afd71dcabdb4636c89d557a3f42db9e2040c8bbd1708aa4ce7
'Hello'    -> 185f8db32271fe25f561a6fc938b2e264306ec304eda518007d1764826381969
'hello '   -> 5e3235a8346e5a4585f8c58562f5052b8fe26a3bb122e1e96c76784964dfc461

hello vs hellp : differing bits 131 / 256
hello vs Hello : differing bits 125 / 256
hello vs 'hello ': differing bits 139 / 256


In [ ]:
# NB04-2: the avalanche effect — flip the input bits one at a time and average how much the output changes
import hashlib

msg = b"The Times 03/Jan/2009 Chancellor on brink of second bailout for banks"
base = hashlib.sha256(msg).digest()

def flip_bit(data: bytes, i: int) -> bytes:
    """Flip the i-th bit (counting from 0) of a byte string."""
    b = bytearray(data)
    b[i // 8] ^= 1 << (i % 8)
    return bytes(b)

diffs = []
for i in range(len(msg) * 8):          # flip every bit of the input once (69 bytes × 8 = 552 bits)
    h = hashlib.sha256(flip_bit(msg, i)).digest()
    diffs.append(sum(bin(a ^ b).count("1") for a, b in zip(base, h)))

print("input bits flipped:", len(diffs))
print("output bits changed — min %d, max %d, mean %.1f (of 256 bits)" % (min(diffs), max(diffs), sum(diffs) / len(diffs)))

input bits flipped: 552
output bits changed — min 101, max 150, mean 127.5 (of 256 bits)


In [ ]:
# NB04-3: forging a checksum is immediate; forging a hash takes exhaustive search
import hashlib

def checksum8(data: bytes) -> int:
    """Last digits of the byte sum (mod 256): a checksum for accidental errors."""
    return sum(data) % 256

original = b"Pay to Alice 100"
target = checksum8(original)
forged = bytearray(b"Pay to Bob   100 ")         # change the payee,
forged.append((target - checksum8(forged)) % 256)   # then append one byte to make the last digits match
print("original checksum:", target)
print("forged checksum  :", checksum8(forged), "| forged:", bytes(forged))

def toy_hash16(data: bytes) -> int:
    """Toy hash that uses only the first 16 bits of SHA-256 (not for real use; for explanation only)."""
    return int.from_bytes(hashlib.sha256(data).digest()[:2], "big")

t = toy_hash16(original)
attempts = 0
while True:
    attempts += 1
    candidate = b"Pay to Bob   100 #" + str(attempts).encode()
    if toy_hash16(candidate) == t:
        break
print("attempts to match the 16-bit toy hash:", attempts, "(expected 2^16 = 65,536)")
print("expected attempts to match all 256 bits: 2^256 ≈ %.1e" % 2**256)

original checksum: 220
forged checksum  : 220 | forged: b'Pay to Bob   100 k'
attempts to match the 16-bit toy hash: 27882 (expected 2^16 = 65,536)
expected attempts to match all 256 bits: 2^256 ≈ 1.2e+77


In [ ]:
# NB04-4: the birthday paradox — how many draws until the first collision in the 16-bit toy hash
import hashlib, math

def toy_hash16(data: bytes) -> int:
    return int.from_bytes(hashlib.sha256(data).digest()[:2], "big")

def draws_until_collision(run: int) -> int:
    """Experiment number run: feed in distinct inputs one at a time and count until a hash repeats for the first time."""
    seen = {}
    k = 0
    while True:
        k += 1
        h = toy_hash16(f"run{run}-item{k}".encode())
        if h in seen:
            return k
        seen[h] = k

results = [draws_until_collision(r) for r in range(200)]
N = 2**16
print("runs 200, output space N = %d" % N)
print("draws until first collision — mean %.1f, min %d, max %d" % (sum(results) / len(results), min(results), max(results)))
print("theoretical expectation sqrt(pi*N/2) = %.1f" % math.sqrt(math.pi * N / 2))
print("n at which collision probability is 1/2: n ≈ 1.18*sqrt(N) = %.0f" % (1.18 * math.sqrt(N)))
print("for SHA-256 (N = 2^256), 1.18*sqrt(N) ≈ %.2e" % (1.18 * math.sqrt(2**256)))

runs 200, output space N = 65536
draws until first collision — mean 325.1, min 19, max 754
theoretical expectation sqrt(pi*N/2) = 320.8
n at which collision probability is 1/2: n ≈ 1.18*sqrt(N) = 302
for SHA-256 (N = 2^256), 1.18*sqrt(N) ≈ 4.02e+38


In [ ]:
# NB04-5: SHA-256 by hand — seeing the iterated structure of the standard's specification (for study, not for real use)
import hashlib, struct

def icbrt(n: int) -> int:
    """Integer cube root (rounded down) — uses integer arithmetic only, so the constants carry no floating-point error."""
    x = 1 << ((n.bit_length() + 2) // 3)
    while True:
        y = (2 * x + n // (x * x)) // 3
        if y >= x:
            return x
        x = y

PRIMES = [2,3,5,7,11,13,17,19,23,29,31,37,41,43,47,53,59,61,67,71,73,79,83,89,97,101,103,107,109,113,
          127,131,137,139,149,151,157,163,167,173,179,181,191,193,197,199,211,223,227,229,233,239,241,251,
          257,263,269,271,277,281,283,293,307,311]
# constants K: first 32 bits of the fractional parts of the cube roots of the first 64 primes (FIPS 180-4 §4.2.2) — the integer part of cbrt(p)·2^32, modulo 2^32
K = [icbrt(p << 96) & 0xFFFFFFFF for p in PRIMES]
# initial hash value H0: first 32 bits of the fractional parts of the square roots of the first 8 primes (FIPS 180-4 §5.3.3)
import math
H0 = [math.isqrt(p << 64) & 0xFFFFFFFF for p in PRIMES[:8]]

def rotr(x, n): return ((x >> n) | (x << (32 - n))) & 0xFFFFFFFF

def compress(H, block):
    """Compression function: 256-bit state H and one 512-bit block → new state (FIPS 180-4 §6.2.2)"""
    w = list(struct.unpack(">16I", block))
    for t in range(16, 64):
        s0 = rotr(w[t-15], 7) ^ rotr(w[t-15], 18) ^ (w[t-15] >> 3)
        s1 = rotr(w[t-2], 17) ^ rotr(w[t-2], 19) ^ (w[t-2] >> 10)
        w.append((w[t-16] + s0 + w[t-7] + s1) & 0xFFFFFFFF)
    a, b, c, d, e, f, g, h = H
    for t in range(64):
        S1 = rotr(e, 6) ^ rotr(e, 11) ^ rotr(e, 25)
        ch = (e & f) ^ (~e & g)
        T1 = (h + S1 + ch + K[t] + w[t]) & 0xFFFFFFFF
        S0 = rotr(a, 2) ^ rotr(a, 13) ^ rotr(a, 22)
        maj = (a & b) ^ (a & c) ^ (b & c)
        T2 = (S0 + maj) & 0xFFFFFFFF
        h, g, f, e, d, c, b, a = g, f, e, (d + T1) & 0xFFFFFFFF, c, b, a, (T1 + T2) & 0xFFFFFFFF
    return [(x + y) & 0xFFFFFFFF for x, y in zip(H, [a, b, c, d, e, f, g, h])]

def sha256_manual(msg: bytes, trace=False) -> str:
    L = len(msg) * 8
    padded = msg + b"\x80" + b"\x00" * ((55 - len(msg)) % 64) + struct.pack(">Q", L)   # §5.1.1 padding
    H = H0[:]
    for i in range(0, len(padded), 64):
        H = compress(H, padded[i:i+64])
        if trace:
            print(f"  state after block {i//64 + 1}, H^({i//64 + 1}) = {''.join(f'{x:08x}' for x in H)}")
    return "".join(f"{x:08x}" for x in H)

for m in [b"", b"abc", b"hello"]:
    ok = sha256_manual(m) == hashlib.sha256(m).hexdigest()
    print(f"{m!r:9} by hand == hashlib ? {ok}")

two_blocks = b"a" * 100          # 100 bytes → two 512-bit blocks after padding
print("state changes for a 100-byte message (2 blocks):")
digest = sha256_manual(two_blocks, trace=True)
print("  final digest               =", digest)
print("  hashlib                    =", hashlib.sha256(two_blocks).hexdigest())

b''       by hand == hashlib ? True
b'abc'    by hand == hashlib ? True
b'hello'  by hand == hashlib ? True
state changes for a 100-byte message (2 blocks):
  state after block 1, H^(1) = df5bb81ce81e0626fb45a8944fd40f31b25e6816d6d499c1ab90492900635e66
  state after block 2, H^(2) = 2816597888e4a0d3a36b82b83316ab32680eb8f00f8cd3b904d681246d285a0e
  final digest               = 2816597888e4a0d3a36b82b83316ab32680eb8f00f8cd3b904d681246d285a0e
  hashlib                    = 2816597888e4a0d3a36b82b83316ab32680eb8f00f8cd3b904d681246d285a0e
